# 06 - Modellering och utvärdering

## Syfte

Syftet med denna notebook är att träna och utvärdera en modell som
klassificerar om en order riskerar att levereras sent (`late = 1`).

Modelleringen bygger på de preprocessade dataset som skapades i föregående
notebook.

Arbetsflödet är:

- skapa en enkel baseline som jämförelse
- träna en klassificeringsmodell på träningsdata
- använda validation-data för modellval och tuning
- utvärdera med metrics som tar hänsyn till klassobalansen
- använda testdata först när modellvalet är färdigt
- jämföra den slutliga modellen med baseline
- spara den tränade modellen och en sammanfattning av resultaten

Eftersom target är obalanserad används inte accuracy som enda mått.
Särskild vikt läggs på modellens förmåga att identifiera sena leveranser.

In [41]:
from pathlib import Path

import pandas as pd

# Sökväg till artifacts från feature engineering
artifacts_dir = Path("artifacts")

# Läs endast train och validation under modellutvecklingen
train_data = pd.read_csv(artifacts_dir / "model_train.csv")
validation_data = pd.read_csv(artifacts_dir / "model_validation.csv")

# Separera features och target
X_train = train_data.drop(columns="late")
y_train = train_data["late"]

X_validation = validation_data.drop(columns="late")
y_validation = validation_data["late"]

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)

print("\ny_train:", y_train.shape)
print("y_validation:", y_validation.shape)

print("\nSamma feature-kolumner:",
      X_train.columns.equals(X_validation.columns))

print("\nTestdata läses inte in ännu.")

X_train: (67533, 62)
X_validation: (14471, 62)

y_train: (67533,)
y_validation: (14471,)

Samma feature-kolumner: True

Testdata läses inte in ännu.


## Baseline-modell

Som referens används en enkel `DummyClassifier` som alltid predikterar
den vanligaste klassen.

Baseline-modellen visar vilken prestanda som kan uppnås utan att modellen
lär sig några samband mellan features och target.

Eftersom `late` är obalanserad räcker accuracy inte som utvärderingsmått.
Därför jämförs även precision, recall, F1-score, balanced accuracy och
average precision för den positiva klassen (`late = 1`).

In [42]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    average_precision_score
)

# Träna en enkel baseline på träningsdata
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Utvärdera baseline på validation-data
baseline_pred = baseline.predict(X_validation)
baseline_prob = baseline.predict_proba(X_validation)[:, 1]

print("Baseline på validation:")
print(f"Accuracy:           {accuracy_score(y_validation, baseline_pred):.4f}")
print(f"Precision:          {precision_score(y_validation, baseline_pred, zero_division=0):.4f}")
print(f"Recall:             {recall_score(y_validation, baseline_pred, zero_division=0):.4f}")
print(f"F1-score:           {f1_score(y_validation, baseline_pred, zero_division=0):.4f}")
print(f"Balanced accuracy:  {balanced_accuracy_score(y_validation, baseline_pred):.4f}")
print(f"Average precision:  {average_precision_score(y_validation, baseline_prob):.4f}")

Baseline på validation:
Accuracy:           0.9466
Precision:          0.0000
Recall:             0.0000
F1-score:           0.0000
Balanced accuracy:  0.5000
Average precision:  0.0534


### Tolkning av baseline

Baseline-modellen uppnår en accuracy på 94,66 %, men detta resultat är
missvisande eftersom modellen alltid predikterar majoritetsklassen
`late = 0`.

Modellen identifierar därför inga sena leveranser, vilket ger både recall
och F1-score på 0 för den positiva klassen. Balanced accuracy är 0,50,
vilket motsvarar en modell utan användbar förmåga att skilja mellan
klasserna.

Average precision är 0,0534, ungefär i nivå med andelen sena leveranser
i validation-data.

Resultatet visar att en användbar modell måste utvärderas med metrics som
tar hänsyn till minoritetsklassen och inte enbart med accuracy.

## Logistisk regression

Som första klassificeringsmodell används logistisk regression.

Modellen är relativt enkel och lätt att tolka, vilket gör den lämplig som
ett första steg efter baseline-modellen.

Eftersom target är obalanserad används `class_weight="balanced"`.
Det innebär att modellen ger större vikt åt minoritetsklassen `late = 1`
under träningen.

Regulariseringsparametern `C` testas med flera värden. Modellerna tränas
endast på träningsdata och jämförs på validation-data.

`Average Precision` används som primärt mått vid modellval eftersom den
positiva klassen är relativt ovanlig. Även balanced accuracy, precision,
recall och F1-score följs för att ge en mer komplett bild av modellens
prestanda.

Testdata används inte under detta modellval.

In [43]:
from sklearn.linear_model import LogisticRegression

# Testa några värden för regulariseringsparametern C
c_values = [0.01, 0.1, 1, 10]

tuning_results = []

for c in c_values:
    model = LogisticRegression(
        C=c,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

    # Träna endast på träningsdata
    model.fit(X_train, y_train)

    # Utvärdera på validation-data
    validation_pred = model.predict(X_validation)
    validation_prob = model.predict_proba(X_validation)[:, 1]

    result = {
        "C": c,
        "accuracy": accuracy_score(y_validation, validation_pred),
        "precision": precision_score(
            y_validation,
            validation_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            validation_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            validation_pred,
            zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_validation,
            validation_pred
        ),
        "average_precision": average_precision_score(
            y_validation,
            validation_prob
        )
    }

    tuning_results.append(result)

# Visa resultaten som tabell
tuning_results_df = pd.DataFrame(tuning_results)

print(tuning_results_df.round(4))

       C  accuracy  precision  recall      f1  balanced_accuracy  \
0   0.01    0.6867     0.1144  0.7219  0.1975             0.7033   
1   0.10    0.6870     0.1144  0.7206  0.1974             0.7029   
2   1.00    0.6870     0.1142  0.7193  0.1971             0.7022   
3  10.00    0.6865     0.1141  0.7193  0.1969             0.7020   

   average_precision  
0             0.1595  
1             0.1630  
2             0.1637  
3             0.1639  


### Resultat av modellval

Samtliga logistiska regressionsmodeller presterar tydligt bättre än
baseline-modellen när förmågan att identifiera sena leveranser beaktas.

Det högsta värdet för Average Precision erhålls med `C = 10`, där
Average Precision är 0,1639 jämfört med 0,0534 för baseline.

Modellen med `C = 10` identifierar cirka 72 % av de sena leveranserna
(recall = 0,7193), medan baseline inte identifierade några sena
leveranser alls.

Precision är samtidigt relativt låg, cirka 0,114. Det innebär att många
ordrar som klassificeras som sena i verkligheten inte är sena. Detta är
en konsekvens av avvägningen mellan precision och recall när modellen
tränas med `class_weight="balanced"` på ett obalanserat dataset.

Skillnaderna mellan de testade värdena på `C` är små, men eftersom
Average Precision används som primärt mått väljs `C = 10` som
slutlig modellkonfiguration.

Testdata har fortfarande inte använts.

In [44]:
# Träna den valda slutliga modellen på träningsdata
final_model = LogisticRegression(
    C=10,
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

final_model.fit(X_train, y_train)

print("Slutlig modell tränad.")
print("Modell:", final_model)
print("Antal features:", final_model.n_features_in_)
print("\nTestdata har fortfarande inte använts.")

Slutlig modell tränad.
Modell: LogisticRegression(C=10, class_weight='balanced', max_iter=2000,
                   random_state=42)
Antal features: 62

Testdata har fortfarande inte använts.


## Slutlig utvärdering på testdata

Modellval och tuning är nu avslutade.

Testdata läses in först i detta steg och används endast för den slutliga
utvärderingen. Testresultatet används alltså inte för att ändra modell,
features eller hyperparametrar.

Detta ger en mer rättvis uppskattning av hur modellen fungerar på data
som inte har använts under modellutvecklingen.

In [45]:
# Läs testdata först efter avslutat modellval
test_data = pd.read_csv(artifacts_dir / "model_test.csv")

X_test = test_data.drop(columns="late")
y_test = test_data["late"]

# Slutliga prediktioner på testdata
test_pred = final_model.predict(X_test)
test_prob = final_model.predict_proba(X_test)[:, 1]

# Beräkna slutliga metrics
test_accuracy = accuracy_score(y_test, test_pred)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
test_balanced_accuracy = balanced_accuracy_score(y_test, test_pred)
test_average_precision = average_precision_score(y_test, test_prob)

print("Slutlig modell på testdata:")
print(f"Accuracy:           {test_accuracy:.4f}")
print(f"Precision:          {test_precision:.4f}")
print(f"Recall:             {test_recall:.4f}")
print(f"F1-score:           {test_f1:.4f}")
print(f"Balanced accuracy:  {test_balanced_accuracy:.4f}")
print(f"Average precision:  {test_average_precision:.4f}")

Slutlig modell på testdata:
Accuracy:           0.4684
Precision:          0.0970
Recall:             0.8474
F1-score:           0.1741
Balanced accuracy:  0.6445
Average precision:  0.1145


### Tolkning av testresultatet

Den slutliga modellen uppnår en recall på 0,8474 på testdata, vilket innebär
att modellen identifierar en stor andel av de sena leveranserna.

Precision är däremot låg, 0,0970. Modellen klassificerar alltså många
ordrar som sena trots att de inte blir sena. Detta bidrar också till den
relativt låga accuracy-nivån på 0,4684.

Balanced accuracy är 0,6445 och Average Precision är 0,1145. Average
Precision är lägre än på validation-data (0,1639), vilket visar att
modellens prestanda försämras på den senare testperioden.

Eftersom datasetet delades kronologiskt representerar testdata en senare
tidsperiod än tränings- och validation-data. Skillnaden mellan validation-
och testresultatet kan därför tyda på att sambanden i data förändras över
tid.

Trots försämringen identifierar modellen betydligt fler sena leveranser än
baseline-modellen. Samtidigt visar den låga precisionen att modellen i sin
nuvarande form genererar många falska positiva prediktioner och därför har
tydliga begränsningar.

Testresultatet används endast som slutlig utvärdering. Modellen eller dess
hyperparametrar ändras inte utifrån testresultatet.

In [46]:
# Hämta validation-resultat för den valda modellen C=10
selected_validation = tuning_results_df.loc[
    tuning_results_df["C"] == 10
].iloc[0]

# Jämför baseline och vald modell på samma validation-data
comparison_df = pd.DataFrame({
    "Modell": [
        "Baseline",
        "Logistisk regression (C=10)"
    ],
    "Accuracy": [
        accuracy_score(y_validation, baseline_pred),
        selected_validation["accuracy"]
    ],
    "Precision": [
        precision_score(
            y_validation, baseline_pred, zero_division=0
        ),
        selected_validation["precision"]
    ],
    "Recall": [
        recall_score(
            y_validation, baseline_pred, zero_division=0
        ),
        selected_validation["recall"]
    ],
    "F1": [
        f1_score(
            y_validation, baseline_pred, zero_division=0
        ),
        selected_validation["f1"]
    ],
    "Balanced accuracy": [
        balanced_accuracy_score(y_validation, baseline_pred),
        selected_validation["balanced_accuracy"]
    ],
    "Average precision": [
        average_precision_score(y_validation, baseline_prob),
        selected_validation["average_precision"]
    ]
})

print(comparison_df.round(4).to_string(index=False))

                     Modell  Accuracy  Precision  Recall     F1  Balanced accuracy  Average precision
                   Baseline    0.9466     0.0000  0.0000 0.0000              0.500             0.0534
Logistisk regression (C=10)    0.6865     0.1141  0.7193 0.1969              0.702             0.1639


In [47]:
import json
import joblib

# Spara den slutliga tränade modellen
joblib.dump(
    final_model,
    artifacts_dir / "late_delivery_model.joblib"
)

# Spara resultaten från tuning
tuning_results_df.to_csv(
    artifacts_dir / "model_tuning_results.csv",
    index=False
)

# Samla viktiga resultat
model_results = {
    "model": "LogisticRegression",
    "C": 10,
    "class_weight": "balanced",
    "primary_metric": "average_precision",

    "baseline_validation": {
        "accuracy": float(
            accuracy_score(y_validation, baseline_pred)
        ),
        "precision": float(
            precision_score(
                y_validation,
                baseline_pred,
                zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y_validation,
                baseline_pred,
                zero_division=0
            )
        ),
        "f1": float(
            f1_score(
                y_validation,
                baseline_pred,
                zero_division=0
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_validation,
                baseline_pred
            )
        ),
        "average_precision": float(
            average_precision_score(
                y_validation,
                baseline_prob
            )
        )
    },

    "selected_model_validation": {
        "accuracy": float(selected_validation["accuracy"]),
        "precision": float(selected_validation["precision"]),
        "recall": float(selected_validation["recall"]),
        "f1": float(selected_validation["f1"]),
        "balanced_accuracy": float(
            selected_validation["balanced_accuracy"]
        ),
        "average_precision": float(
            selected_validation["average_precision"]
        )
    },

    "final_test": {
        "accuracy": float(test_accuracy),
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "balanced_accuracy": float(test_balanced_accuracy),
        "average_precision": float(test_average_precision)
    }
}

with open(
    artifacts_dir / "model_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Sparade artifacts:")
print("-", artifacts_dir / "late_delivery_model.joblib")
print("-", artifacts_dir / "model_tuning_results.csv")
print("-", artifacts_dir / "model_results.json")

Sparade artifacts:
- artifacts\late_delivery_model.joblib
- artifacts\model_tuning_results.csv
- artifacts\model_results.json


In [48]:
# Skapa en kort sammanfattning av modellresultaten
summary_text = f"""
MODELLERING - RESULTATSAMMANFATTNING

Baseline på validation:
Accuracy: {accuracy_score(y_validation, baseline_pred):.4f}
Recall: {recall_score(y_validation, baseline_pred, zero_division=0):.4f}
F1-score: {f1_score(y_validation, baseline_pred, zero_division=0):.4f}
Balanced accuracy: {balanced_accuracy_score(y_validation, baseline_pred):.4f}
Average precision: {average_precision_score(y_validation, baseline_prob):.4f}

Vald modell:
Logistic Regression
C: 10
Class weight: balanced

Vald modell på validation:
Accuracy: {selected_validation["accuracy"]:.4f}
Precision: {selected_validation["precision"]:.4f}
Recall: {selected_validation["recall"]:.4f}
F1-score: {selected_validation["f1"]:.4f}
Balanced accuracy: {selected_validation["balanced_accuracy"]:.4f}
Average precision: {selected_validation["average_precision"]:.4f}

Slutlig utvärdering på test:
Accuracy: {test_accuracy:.4f}
Precision: {test_precision:.4f}
Recall: {test_recall:.4f}
F1-score: {test_f1:.4f}
Balanced accuracy: {test_balanced_accuracy:.4f}
Average precision: {test_average_precision:.4f}

Modellen valdes med validation-data och Average Precision som primärt mått.
Testdata användes endast en gång för den slutliga utvärderingen.
"""

summary_path = artifacts_dir / "model_summary.txt"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text.strip())

print("Sparad:", summary_path)

Sparad: artifacts\model_summary.txt


## RESULTAT

I denna notebook tränades och utvärderades en klassificeringsmodell för att
identifiera ordrar med risk för sen leverans.

En `DummyClassifier` användes först som baseline. Baseline-modellen fick
hög accuracy på validation-data (0,9466), men identifierade inga sena
leveranser. Recall och F1-score för den positiva klassen blev därför 0,
och Average Precision var 0,0534.

Därefter tränades logistisk regression med `class_weight="balanced"` för
att ta hänsyn till klassobalansen. Regulariseringsparametern `C` jämfördes
på validation-data med Average Precision som primärt mått. Av de testade
värdena gav `C = 10` högst Average Precision, 0,1639, och valdes därför
som slutlig modellkonfiguration.

På validation-data uppnådde den valda modellen recall 0,7193, balanced
accuracy 0,7020 och Average Precision 0,1639. Modellen identifierade därmed
betydligt fler sena leveranser än baseline, även om precisionen var
relativt låg.

Testdata användes först efter att modellvalet var avslutat. På testdata
uppnådde modellen recall 0,8474, precision 0,0970, F1-score 0,1741,
balanced accuracy 0,6445 och Average Precision 0,1145.

Resultatet visar att modellen har god förmåga att fånga en stor andel av
de sena leveranserna, men samtidigt genererar många falska positiva
prediktioner. Prestandan är också lägre på den senare testperioden än på
validation-data, vilket kan tyda på förändringar i datans mönster över tid.

Testresultatet användes endast för slutlig utvärdering och inga
modellparametrar ändrades utifrån testresultatet.

Följande artifacts sparades:

- `artifacts/late_delivery_model.joblib`
- `artifacts/model_tuning_results.csv`
- `artifacts/model_results.json`
- `artifacts/model_summary.txt`

In [3]:
import sys
print(sys.executable)

c:\Users\aroub\anaconda3\python.exe
